## Token & Param

In [1]:
vocab_size = 50000
dhead = 64
k = 24
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

enc_params = n_blocks * 12 * dmodel ** 2
emb_params = dmodel * vocab_size
params = enc_params + emb_params
print(f'params: {params / 1e6:.0f}M\t encoder: {enc_params/1e6:.0f}M\tembedding: {emb_params/1e6:.0f}M')

dmodel: 1536
params: 756M	 encoder: 679M	embedding: 77M


In [2]:
tokens = None
# tokens = 125e3 * 512 * 512
if tokens is None:
    tokens = 20 * params

ratio = tokens / params
print(f'ratio: {ratio}')
print(f'tokens: {tokens / 1e9:.2f}B')

ratio: 20.0
tokens: 15.13B


In [3]:
batch_size = 512
seq_len = 512
n_steps = tokens / (batch_size * seq_len)
print(f'n steps: {n_steps:.0f}')

n steps: 57699


In [4]:
float_bytes = 4
tokens_memory = float_bytes * dmodel * batch_size * seq_len
print(f'batch_memory: {tokens_memory / 1e6:.0f} MB')

model_memory = 4 * float_bytes * params
print(f'model_memory: {model_memory / 1e9:.0f} GB')

batch_memory: 1611 MB
model_memory: 12 GB


## compute cost $$$

In [5]:
mfu = 0.3
gpu_flops = (1671 / 2) * 1e12

In [6]:
theoretical_flops = 6 * tokens * params
print(f'theoretical_flops: {theoretical_flops / 1e18:.2f}*1E6 TFLOPS')
gpu_hours = theoretical_flops / (gpu_flops * 3600 * mfu)
print(f'GPU hours: {gpu_hours:.1f}')
n_gpus = 8
print(f'training hours: {gpu_hours / n_gpus:.1f}')
print(f'training days: {gpu_hours / (n_gpus * 24):.1f}')

theoretical_flops: 68.63*1E6 TFLOPS
GPU hours: 76.1
training hours: 9.5
training days: 0.4


In [40]:
grid_size = 15
total_gpu_hours = grid_size * gpu_hours
print(f'total_gpu_hours: {total_gpu_hours:.0f}')

total_gpu_hours: 6302


## Calculate MFU

In [7]:
minutes = 30
steps = 1400
batch_size = 512
seq_len = 512
vocab_size = 50000
k = 24
gpu_flops = (1671 / 2) * 1e12
n_gpus = 4
dmodel = dhead * k
print(f'dmodel: {dmodel}')
n_blocks = k

params = n_blocks * 12 * dmodel ** 2 + dmodel * vocab_size
print(f'params: {params / 1e6:.0f}M')

dmodel: 1536
params: 756M


In [8]:
tokens_processed = steps * batch_size * seq_len
th_flops = 6 * tokens_processed * params
real_flops = gpu_flops * n_gpus * 60 * minutes
mfu = th_flops / real_flops
print(f'MFU: {mfu:.3f}')

MFU: 0.277
